In [33]:
from pathlib import Path
import unicodedata
import hashlib
import re

import polars as pl

CSV_PATH = Path("data/researchers.csv")


def load_researcher_ids(csv_path: Path = CSV_PATH) -> list[int]:
    """Load researcher IDs from a CSV after validating format and contents.

    :param csv_path: Path to the CSV file containing researcher data.
    :return: List of researcher IDs.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found at: {csv_path}")

    df = pl.read_csv(csv_path)

    if "idPersonal" not in df.columns:
        raise ValueError(
            f"The CSV does not contain the 'idPersonal' column. Columns found: {df.columns}"
        )

    df = (
        df.select(["idPersonal", "rfSignature"])
        .drop_nulls()
        .filter(pl.col("idPersonal").cast(pl.Int64) > 0)
        .unique()
        .sort("idPersonal")
    )

    ids = df["idPersonal"].to_list()
    names = df["rfSignature"].to_list()
    return ids, names


ids, names = load_researcher_ids()
researchers = dict(zip(ids, names))
researchers

{258423: 'HABER GUERRA, RODOLFO ELIAS',
 258729: 'ROSADO MARIA, MARIA EUGENIA',
 258829: 'VAZQUEZ GALLO, MARIA JESUS',
 260726: 'RICO ALMODOVAR, MARIANO',
 260781: 'LOSADA GONZALEZ, JUAN CARLOS',
 260874: 'PEREZ TEJERO, JAVIER',
 261274: 'CAMACHO FERNANDEZ, DAVID',
 261292: 'MENDO TOMAS, LUIS',
 261323: 'DUQUE CAMPAYO, DANIEL',
 261708: 'RUIZ CRUZ, JORGE ALFONSO',
 262424: 'LOPEZ HERNANDEZ, ISABEL',
 262511: 'SALGADO ALVAREZ DE SOTOMAYOR, LUIS',
 262522: 'FERNANDEZ GONZALEZ, JOSE MANUEL',
 262523: 'BELLO ORGAZ, GEMA',
 262605: 'MARTINEZ SAEZ, ANTONIO',
 263256: 'RAMOS GOMEZ, MARIA MILAGROS',
 264089: 'TORIBIO LOPEZ, FRANCISCO RENE',
 264099: 'ANGULO ARDOY, PABLO',
 264517: 'AGUIRRE DE CARCER GARCIA, IŃIGO',
 264698: 'MERCHAN RUBIRA, SUSANA',
 264784: 'SANCHEZ OLIVARES, PABLO',
 264906: 'GARCIA MUŃOZ, PATRICIA',
 265232: 'GONZALEZ ALVARO, DAVID',
 271169: 'BERROCAL MENARGUEZ, ANA BELEN',
 271668: 'SALAS NATERA, MIGUEL ALEJANDRO',
 271791: 'MEROŃO MORENO, CRISTOBAL',
 272078: 'GALVEZ PER

### Load data

In [2]:
import json
import re
from collections import defaultdict

from scrapy import Selector

ID = 308607  # Carlos: 308607, Camacho: 261274, Luengo: 307275, Andrea: 307657
rf_signature = researchers.get(ID, "Desconocido")
print(f"Investigador: {rf_signature} (ID: {ID})")
BASE_PATH = Path("./data/raw")
MAIN_HTML = BASE_PATH / f"{ID}_main.html"
AJAX_HTML = BASE_PATH / f"{ID}_ajax.html"

html = MAIN_HTML.read_text(encoding="utf-8")
sel = Selector(text=html)

Investigador: BADENES OLMEDO, CARLOS (ID: 308607)


In [3]:
def load_gantt_data(html: str) -> list[dict]:
    """Load Gantt chart data from the provided HTML content.

    :param html: HTML content containing the Gantt chart data.
    :return: List of dictionaries representing the Gantt chart data.
    """
    try:
        gantt_data = re.search(
            r"const ganttData\s*=\s*(\[.*?\]);\s*//", html, re.DOTALL
        ).group(1)
    except Exception as e:
        print(f"Error: {e}")
        gantt_data = "[]"
    return json.loads(gantt_data)


def create_researcher_df(gantt_data: list[dict]) -> pl.DataFrame:
    df = pl.DataFrame(gantt_data)

    df = df.select(
        [
            pl.col("rol").fill_null("-"),
            pl.col("label_event").alias("evento"),
            pl.col("entity").alias("entidad"),
            pl.col("duration")
            .str.extract(r"(\d{4}-\d{2}-\d{2})", 1)
            .str.to_date()
            .alias("fecha_inicio"),
            pl.col("duration")
            .str.extract(r"/\s*(\d{4}-\d{2}-\d{2})", 1)
            .str.to_date()
            .alias("fecha_fin"),
        ]
    )

    df = df.with_columns(pl.col("fecha_fin").fill_null(pl.col("fecha_inicio")))

    return df


gantt = load_gantt_data(html)

for item in gantt:
    rol = item.get("rol") if item.get("rol") is not (None or "") else None
    evento = (
        item.get("label_event") if item.get("label_event") is not (None or "") else None
    )
    duration = (
        item.get("duration") if item.get("duration") is not (None or "") else None
    )
    entidad = item.get("entity") if item.get("entity") is not (None or "") else None
    print(f"Rol: {rol} | Evento: {evento} | Duración: {duration} | Entidad: {entidad}")

if not gantt:
    print("No se encontraron datos de Gantt en el HTML.")
else:
    researcher_trajectory = create_researcher_df(gantt)
    researcher_trajectory.schema

Rol: Investigador | Evento: INVESTIGADOR | Duración: 2025-04-12 / 2026-03-07 | Entidad: Universidad Politécnica de Madrid. Centro de I+D+i en Inteligencia Artificial (AI.nnovation Space)
Rol: Profesor Ayudante Doctor | Evento: PROFESOR AYUDANTE DOCTOR | Duración: 2022-09-01 / 2026-08-31 | Entidad: SISTEMAS INFORMÁTICOS. Universidad Politécnica de Madrid
Rol: L.D. PRF.AYUD.DOCTOR. SISTEMAS INFORMÁTICOS | Evento: L.D. PRF.AYUD.DOCTOR | Duración: 2022-09-01 / 2024-02-01 | Entidad: Universidad Politécnica de Madrid 
Rol: Miembro | Evento: MIEMBRO | Duración: 2015-10-22 / 2026-03-07 | Entidad: Universidad Politécnica de Madrid. Ontology Engineering Group
Rol: Software Engineer. AMPLIA SOLUCIONES, S.L. | Evento: SOFTWARE ENGINEER | Duración: 2006-04-01 / 2014-12-31 | Entidad: AMPLIA SOLUCIONES, S.L.
Rol: Último sexenio concedido | Evento: SEXENIO | Duración: 2024-01-01 / 2024-01-01 | Entidad: None
Rol: Primer Proyecto como IP | Evento: IP PROYECTO | Duración: 2023-12-01 / 2023-12-01 | Entida

In [4]:
def load_json_data(html: str) -> dict:
    """Load JSON data from the provided HTML content.

    :param html: HTML content containing the JSON data.
    :return: Two dictionaries representing the loaded JSON data.
    """
    try:
        df1 = pl.DataFrame(
            json.loads(
                re.search(r"var json1\s*=\s*(\[.*?\]);", html, re.DOTALL).group(1)
            )
        ).rename({"parent": "type", "Begin": "year_start"})
        df2 = pl.DataFrame(
            json.loads(
                re.search(r"var json2\s*=\s*(\[.*?\]);", html, re.DOTALL).group(1)
            )
        ).rename({"Type": "type2", "parent": "year_start2"})

        # titles_match = set(df1["Title"].to_list()) == set(df2["Title"].to_list())

        df = df1.join(
            df2.select(["Title", "year_start2", "type2"]), on="Title", how="full"
        )

        # discrepancias_type = df.filter(pl.col("type") != pl.col("type2"))
        # discrepancias_year = df.filter(pl.col("year_start").str.strip_chars() != pl.col("year_start2").str.strip_chars())
        # print(f"Discrepancias en type: {len(discrepancias_type)}")
        # print(f"Discrepancias en year_start: {len(discrepancias_year)}")

        df = df.drop(["type2", "year_start2"])

        df = df.with_columns(
            [
                pl.col("year_start").str.strip_chars(),
                pl.col("End").str.strip_chars().replace("Current", None),
                pl.col("itemtype").cast(pl.Int32),
            ]
        )
        df = df.drop("Title_right").with_columns(
            [
                pl.col("year_start").cast(pl.Int16),
                pl.col("End").cast(
                    pl.Int16, strict=False
                ),  # strict=False para tolerar los None
            ]
        )

        print(f"Total records loaded: {len(df)}")

        return df
    except Exception as e:
        print(f"Error loading JSON data: {e}")
        return pl.DataFrame()  # Return an empty DataFrame on error


# Esta info está toda en el AJAX, es redundante cargarla desde el main
proyectos_idi = load_json_data(html)
proyectos_idi.show()

Total records loaded: 31


Title,type,institution,itemtype,year_start,End
str,str,str,i32,i16,i16
"""Promoting Scientific Creativit…","""Proyecto Competitivo""","""Ansmart LTD (Participant),Bour…",50,2014,2017
"""A Platform for Apps and Servic…","""Proyecto Competitivo""","""CEFRIEL - SOCIETA CONSORTILE A…",50,2015,2015
"""3cixty A Platform for Apps and…","""Proyecto Competitivo""","""EIT ICT Labs-14523; Unión Euro…",50,2015,2016
"""DrInventor: Promoting Scientif…","""Proyecto Competitivo""","""Seventh Framework Programme of…",50,2015,2017
"""Datos 4.0: Retos y Soluciones …","""Proyecto Competitivo""","""Ministerio de Economia y Compe…",50,2016,2021


In [5]:
import polars.selectors as cs


def load_ajax_dataSet(file):
    with open(file, encoding="utf-8") as f:
        ajax_html = f.read()

    match = re.search(r"var dataSet\s*=\s*(\[.*\]);", ajax_html, re.DOTALL)

    if not match:
        return pl.DataFrame()

    try:
        json_str = match.group(1)
        activity_data = json.loads(json_str)
    except json.JSONDecodeError:
        try:
            json_str = json_str.rsplit("]", 1)[0] + "]"
            activity_data = json.loads(json_str)
        except Exception as e:
            print(f"Error parsing JSON: {e}")
            return pl.DataFrame()

    if not activity_data:
        return pl.DataFrame()

    all_fields = set().union(*(record.keys() for record in activity_data))
    normalized_records = [
        {f: record.get(f) for f in all_fields} for record in activity_data
    ]

    df = pl.DataFrame(normalized_records, infer_schema_length=len(normalized_records))

    mapeo_columnas = {
        "rfTitle": "ACTIVIDAD",
        "rfAuthor": "AUTORES",
        "rfKeyword": "KEYWORDS",
        # "dsDescription": "RESUMEN",
        "rfStartDate": "FECHA INICIO",
        "rfEndDate": "FECHA FIN",
        "rfHash": "IDENTIFICADOR",
        # "stOpenAccess": "OPENACCESS",
        "dtWorkform": "TIPO",
        "dtType": "SUBTIPO",
        # "rfSubject": "CATEGORÍAS",  <= La usaremos cuando la columna `KEYWORDS` esté vacía
        # "rfLinkName": "FUENTE",
        # "rfQuality11": "IF SJR",
        # "rfQuality17": "CITAS EUROPEPMC",
        # "rfQuality18": "CITAS INSPIRE",
        # "rfQuality19": "CITAS SCHOLAR",
        # "rfQuality22": "Q SJR",
        # "rfRef": "CITACIÓN",
        # "idQuality63": "PIDs",
        "rfRefUrl": "URL",
        "idItem": "PORTAL_UPM_ID",
    }

    columnas_presentes = [c for c in mapeo_columnas.keys() if c in df.columns]
    if not columnas_presentes or "dtWorkform" not in columnas_presentes:
        return pl.DataFrame()

    mapeo_final = {k: mapeo_columnas[k] for k in columnas_presentes}

    df_final = df.select(columnas_presentes).rename(mapeo_final)

    df_final = df_final.with_columns(
        cs.string()
        .str.replace_all(r"[\r\n]+", "; ")
        .str.replace_all(r";", ",")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars('"')
        .str.strip_chars()
    )

    if "URL" in df_final.columns:
        df_final = df_final.with_columns(
            pl.col("URL").map_elements(
                lambda s: ", ".join(re.findall(r"https?://\S+", s)) if s else s,
                return_dtype=pl.String,
            )
        )

    df_final = df_final.sort(
        [c for c in ["TIPO", "SUBTIPO", "FECHA INICIO"] if c in df_final.columns],
        descending=False,
    )

    df_final.write_csv(f"ajax_{ID}_limpio.csv")

    return df_final


df_ajax = load_ajax_dataSet(AJAX_HTML)
df_ajax.select(pl.col("TIPO").value_counts()).to_series().to_list()

[{'TIPO': 'Breve Presentación', 'count': 1},
 {'TIPO': 'Dirección de Tesis, TFG, TFM, etc.', 'count': 52},
 {'TIPO': 'Docencia Impartida', 'count': 29},
 {'TIPO': 'Estancias', 'count': 2},
 {'TIPO': 'Eventos para la Divulgación', 'count': 5},
 {'TIPO': 'Experiencia en Evaluación', 'count': 4},
 {'TIPO': 'Experiencia profesional', 'count': 8},
 {'TIPO': 'Formación', 'count': 19},
 {'TIPO': 'Gestión', 'count': 4},
 {'TIPO': 'Grupos, Redes, Sociedades, Foros y Comités', 'count': 25},
 {'TIPO': 'Idiomas', 'count': 1},
 {'TIPO': 'Otros Resultados Transferidos', 'count': 2},
 {'TIPO': 'Perfil Temático', 'count': 1},
 {'TIPO': 'Propiedad Industrial e Intelectual', 'count': 3},
 {'TIPO': 'Proyectos I+D+I y Ayudas', 'count': 29},
 {'TIPO': 'Proyectos de Innovacion Docente', 'count': 4},
 {'TIPO': 'Proyectos en Ámbito de Salud', 'count': 2},
 {'TIPO': 'Publicaciones', 'count': 69},
 {'TIPO': 'Publicaciones Docentes', 'count': 3},
 {'TIPO': 'Publicaciones de Divulgación', 'count': 1},
 {'TIPO': '

## GLOBAL

### Datos personales

In [6]:
title_text = sel.css("title::text").get("")
full_name = title_text.split(" - ")[0].strip().title() if " - " in title_text else None

first_name, last_name = None, None
if rf_signature and "," in rf_signature:
    parts = rf_signature.split(",", 1)
    last_name = parts[0].strip().title()
    first_name = parts[1].strip().title()

email = sel.xpath('//a[contains(@class,"fa-envelope")]/@data-original-title').get()

print(f"Nombre: {full_name}")
print(f"Nombre por partes: {first_name} - {last_name}")
print(f"Email: {email}")

Nombre: Carlos Badenes Olmedo
Nombre por partes: Carlos - Badenes Olmedo
Email: carlos.badenes@upm.es


### Situación actual

In [7]:
situacion_actual = {
    event.xpath("normalize-space(./div[1]/text())").get(): {
        "entidad": event.xpath("normalize-space(./a/text())").get(),
        "rol": event.xpath("normalize-space(./div[2]/text())").get(),
        "fecha": event.xpath("normalize-space(./div[3]/text())").get(),
    }
    for event in sel.xpath(
        '//div[contains(@class,"ipublic-panel")]//div[@class="event"]'
    )
}
print(f"Situación actual: {json.dumps(situacion_actual, indent=4, ensure_ascii=False)}")
# ^^ Si se repiten dos entidades, se sobreescribe una por otra, como en David Luengo. ^^
# Creo que es porque una de las entradas es cuando entró en el Departamento y el otro campo es porque ahora es el Director del Departamento
# Como lo que me interesa es simplmente la situación actual, no hay problema por sobreescribir el campo más actual (p. ej. Director)

Situación actual: {
    "Grupo de investigacion": {
        "entidad": "Ontology Engineering Group",
        "rol": "Miembro",
        "fecha": "2015-10-22"
    },
    "Instituto o centro Universitario de Investigación": {
        "entidad": "Centro de I+D+i en Inteligencia Artificial (AI.nnovation Space)",
        "rol": "Investigador",
        "fecha": "2025-04-12"
    },
    "Departamento Universitario": {
        "entidad": "SISTEMAS INFORMÁTICOS",
        "rol": "Profesor Permanente Laboral",
        "fecha": "2024-02-02"
    }
}


### Identificadores

In [8]:
def extract_id_from_title(sel, href_fragment):
    """Extrae el ID que aparece tras ' - ' en el atributo data-original-title."""
    title = sel.xpath(
        f'//a[contains(@href, "{href_fragment}")]/@data-original-title'
    ).get()
    if title and " - " in title:
        return title.split(" - ", 1)[1].strip()
    return None


ids_researcher = {
    "investigador": rf_signature,
    "id_interno": ID,
    "email": sel.xpath(
        '//a[contains(@class,"fa-envelope")]/@data-original-title'
    ).get(),
    "web_personal": sel.xpath(
        '//a[contains(@class,"fa-external-link") or @title[starts-with(., "http")]]/@href'
    ).get(),
    "orcid_url": sel.xpath('//a[contains(@href,"orcid.org")]/@href').get(),
    "orcid_id": extract_id_from_title(sel, "orcid.org"),
    "researcher_id_url": sel.xpath(
        '//a[contains(@href,"webofscience.com")]/@href'
    ).get(),
    "researcher_id": extract_id_from_title(sel, "webofscience.com"),
    "scopus_url": sel.xpath('//a[contains(@href,"scopus.com")]/@href').get(),
    "scopus_id": extract_id_from_title(sel, "scopus.com"),
    "google_scholar_url": sel.xpath(
        '//a[contains(@href,"scholar.google")]/@href'
    ).get(),
    "google_scholar_id": extract_id_from_title(sel, "scholar.google"),
    "researchgate_url": sel.xpath(
        '//a[contains(@href,"researchgate.net")]/@href'
    ).get(),
    "researchgate_id": extract_id_from_title(sel, "researchgate.net"),
    "dialnet_url": sel.xpath('//a[contains(@href,"dialnet.unirioja.es")]/@href').get(),
    "dialnet_id": extract_id_from_title(sel, "dialnet.unirioja.es"),
    "mendeley_url": sel.xpath('//a[contains(@class,"bd-mendeley")]/@href').get(),
    "linkedin_url": sel.xpath('//a[contains(@class,"bd-linkedin")]/@href').get(),
}

ids_researcher = {
    k: (None if v in ("", "https://", "http://") else v)
    for k, v in ids_researcher.items()
}

print(
    f"Identificadores del investigador: {json.dumps(ids_researcher, indent=4, ensure_ascii=False)}"
)

Identificadores del investigador: {
    "investigador": "BADENES OLMEDO, CARLOS",
    "id_interno": 308607,
    "email": "carlos.badenes@upm.es",
    "web_personal": "https://about.me/cbadenes",
    "orcid_url": "http://orcid.org/0000-0002-2753-9917",
    "orcid_id": "0000-0002-2753-9917",
    "researcher_id_url": "https://www.webofscience.com/wos/author/rid/GRE-8010-2022",
    "researcher_id": "GRE-8010-2022",
    "scopus_url": "https://www.scopus.com/authid/detail.uri?authorId=57195960598&origin=inward&txGid",
    "scopus_id": "57195960598",
    "google_scholar_url": "https://scholar.google.es/citations?hl=es-ES&user=7U87QYEAAAAJ&hl=es",
    "google_scholar_id": "7U87QYEAAAAJ&hl=es",
    "researchgate_url": "https://www.researchgate.net/profile/Carlos-Badenes-Olmedo",
    "researchgate_id": "Carlos-Badenes-Olmedo",
    "dialnet_url": null,
    "dialnet_id": null,
    "mendeley_url": null,
    "linkedin_url": "https://www.linkedin.com/in/cbadenes"
}


### Extras

In [9]:
print(
    f"Idiomas: {df_ajax.filter(pl.col('TIPO') == 'Idiomas').select('ACTIVIDAD').to_dicts()}"
)
print(
    f"Breve descripción: {df_ajax.filter(pl.col('TIPO') == 'Breve descripción').select('ACTIVIDAD').to_dicts()}"
)
# print(f'Perfil Temático: {df_ajax.filter(pl.col("TIPO")=="Perfil Temático").select("ACTIVIDAD").to_dicts()}')
print(
    f"Competencias: {df_ajax.filter(pl.col('TIPO') == 'Competencias').select('ACTIVIDAD').to_dicts()}"
)

Idiomas: [{'ACTIVIDAD': 'Inglés'}]
Breve descripción: []
Competencias: []


### Temáticas

In [10]:
def normalize_topic(topic: str) -> str:
    """Normaliza una temática para facilitar el conteo y comparación.

    El proceso incluye:
        (1) Convertir a minúsculas y eliminar espacios al inicio y final.
        (2) Eliminar puntuación suelta al final (p.ej., "graphs." -> "graphs").
        (3) Convertir plurales simples a singular (p.ej., "graphs" -> "graph"), pero sin romper palabras como "process".
        (4) Colapsar múltiples espacios en uno solo.

    :param topic: La temática original a normalizar.
    :return: La temática normalizada.
    """
    t = topic.lower().strip()
    t = t.rstrip(".")
    t = re.sub(r"(?<=[a-z]{4})s\b", "", t)
    t = re.sub(r"\s+", " ", t)
    return t


match = re.search(r"var data1\s*=\s*JSON\.parse\('(.*?)'\);", html, re.DOTALL)
data1 = []
if match:
    try:
        data1 = json.loads(match.group(1))
    except json.JSONDecodeError as e:
        print(f"Error parsing data1: {e}")

normalized_counts = defaultdict(int)
for item in data1:
    group = item.get("group", [])
    if len(group) >= 2:
        kw = normalize_topic(group[1])
        normalized_counts[kw] += item.get("current", {}).get("count", 0)

tematicas = sorted(normalized_counts, key=lambda k: normalized_counts[k], reverse=True)
tematicas

['knowledge graph',
 'document retrieval',
 'natural language processing',
 'public procurement',
 'linked data',
 'ontology',
 'antiretroviral drug',
 'hiv',
 'polypharmacy',
 'clustering',
 'document similarity',
 'hashing',
 'information search and retrieval',
 'open data',
 'question answering',
 'topic model',
 'active learning',
 'adaptive learning',
 'analytical model',
 'annotated corpora',
 'api',
 'assessment system',
 'bag-of-word representation',
 'biomedical text processing']

### Publicaciones de divulgación

In [11]:
df_ajax.filter(pl.col("TIPO") == "Publicaciones de Divulgación").select(
    pl.col("ACTIVIDAD"), pl.col("FECHA INICIO"), pl.col("URL")
).to_dicts()

[{'ACTIVIDAD': 'Entrevista en comunidad NLP Spain',
  'FECHA INICIO': '2021-01-28',
  'URL': None}]

### Hitos

In [12]:
def print_milestone(milestone: list[dict]):
    if len(milestone) == 0:
        return

    for row in milestone:
        for key, value in row.items():
            if value is not None and value != "":
                print(f" · {key}: {value}")


try:
    ultimo_sexenio = researcher_trajectory.filter(
        pl.col("evento") == "SEXENIO"
    ).to_dicts()
    primer_q1 = researcher_trajectory.filter(pl.col("evento") == "PRIMER Q1").to_dicts()
    ip_proyecto = researcher_trajectory.filter(
        pl.col("evento") == "IP PROYECTO"
    ).to_dicts()
    doctorado = researcher_trajectory.filter(pl.col("evento") == "DOCTORADO").to_dicts()

    # En el HTML viene "Tramos de investigación", así que de ahí podemos sacar el número total, no solo poner cuándo tuvo el último
    print("Último sexenio:")
    print_milestone(ultimo_sexenio)
    print("\nPrimer Q1:")
    print_milestone(primer_q1)
    print("\nProyecto como Investigador Principal:")
    print_milestone(ip_proyecto)
    print("\nDoctorado:")
    print_milestone(doctorado)
except Exception as e:
    print(f"Error al imprimir hitos: {e}")

Último sexenio:
 · rol: Último sexenio concedido
 · evento: SEXENIO
 · fecha_inicio: 2024-01-01
 · fecha_fin: 2024-01-01

Primer Q1:
 · rol: Publicación de impacto
 · evento: PRIMER Q1
 · entidad: Communications Of The Acm. 18 
 · fecha_inicio: 2016-01-01
 · fecha_fin: 2016-01-01

Proyecto como Investigador Principal:
 · rol: Primer Proyecto como IP
 · evento: IP PROYECTO
 · entidad: Public Procurement Assessment in the Healthcare Sector
 · fecha_inicio: 2023-12-01
 · fecha_fin: 2023-12-01

Doctorado:
 · evento: DOCTORADO
 · entidad: Universidad Politécnica de Madrid
 · fecha_inicio: 2021-06-22
 · fecha_fin: 2021-06-22


In [13]:
# No sé si son mejores que en la celda anterior, al menos estas son más formales (aunque por ejemplo falta primer Q1)
HITOS_RELEVANTES = {
    ("Formación", "Doctorado"),
    ("Formación", "Máster"),
    ("Formación", "Titulado superior / Grado"),
    ("Reconocimientos", None),
    ("Experiencia profesional", "Situación Profesional Actual (Otras Instituciones)"),
}
CAMPOS_HITO = ["ACTIVIDAD", "SUBTIPO", "FECHA INICIO", "URL"]


def is_hito(tipo: str, tipo_produccion: str) -> bool:
    return (tipo, tipo_produccion) in HITOS_RELEVANTES or (
        tipo,
        None,
    ) in HITOS_RELEVANTES


df_hitos = (
    df_ajax.filter(
        pl.struct(["TIPO", "SUBTIPO"]).map_elements(
            lambda x: is_hito(x["TIPO"], x["SUBTIPO"]),
            return_dtype=pl.Boolean,
        )
    )
    .select(CAMPOS_HITO)
    .with_columns(pl.col("FECHA INICIO").str.to_date("%Y-%m-%d", strict=False))
    .sort("FECHA INICIO")
)

df_hitos.sort("FECHA INICIO").to_dicts()

[{'ACTIVIDAD': 'Ingeniero Superior Informático',
  'SUBTIPO': 'Titulado superior / Grado',
  'FECHA INICIO': datetime.date(2008, 11, 27),
  'URL': None},
 {'ACTIVIDAD': 'Máster Universitario de Investigación en Inteligencia Artificial',
  'SUBTIPO': 'Máster',
  'FECHA INICIO': datetime.date(2015, 7, 1),
  'URL': None},
 {'ACTIVIDAD': 'Desafio Aporta 17',
  'SUBTIPO': 'Titulo o Premio a la Calidad Profesional',
  'FECHA INICIO': datetime.date(2017, 10, 31),
  'URL': None},
 {'ACTIVIDAD': '17 Competición actúaupm Mejores Startups 2020',
  'SUBTIPO': 'Titulo o Premio a la Calidad Profesional',
  'FECHA INICIO': datetime.date(2020, 11, 27),
  'URL': None},
 {'ACTIVIDAD': 'Doctorado en Inteligencia Artificial',
  'SUBTIPO': 'Doctorado',
  'FECHA INICIO': datetime.date(2021, 6, 22),
  'URL': 'https://oa.upm.es/67594/'},
 {'ACTIVIDAD': 'DAAD AInet Fellows and Alumni Network',
  'SUBTIPO': 'Ayuda, Beca o contrato de Reconocido Prestigio',
  'FECHA INICIO': datetime.date(2021, 8, 20),
  'URL': 

## INVESTIGACIÓN

### Métricas

In [14]:
# Pasamos de métricas, al menos por el momento

In [15]:
df_ajax.select(pl.col("TIPO").value_counts()).to_series().to_list()

[{'TIPO': 'Breve Presentación', 'count': 1},
 {'TIPO': 'Dirección de Tesis, TFG, TFM, etc.', 'count': 52},
 {'TIPO': 'Docencia Impartida', 'count': 29},
 {'TIPO': 'Estancias', 'count': 2},
 {'TIPO': 'Eventos para la Divulgación', 'count': 5},
 {'TIPO': 'Experiencia en Evaluación', 'count': 4},
 {'TIPO': 'Experiencia profesional', 'count': 8},
 {'TIPO': 'Formación', 'count': 19},
 {'TIPO': 'Gestión', 'count': 4},
 {'TIPO': 'Grupos, Redes, Sociedades, Foros y Comités', 'count': 25},
 {'TIPO': 'Idiomas', 'count': 1},
 {'TIPO': 'Otros Resultados Transferidos', 'count': 2},
 {'TIPO': 'Perfil Temático', 'count': 1},
 {'TIPO': 'Propiedad Industrial e Intelectual', 'count': 3},
 {'TIPO': 'Proyectos I+D+I y Ayudas', 'count': 29},
 {'TIPO': 'Proyectos de Innovacion Docente', 'count': 4},
 {'TIPO': 'Proyectos en Ámbito de Salud', 'count': 2},
 {'TIPO': 'Publicaciones', 'count': 69},
 {'TIPO': 'Publicaciones Docentes', 'count': 3},
 {'TIPO': 'Publicaciones de Divulgación', 'count': 1},
 {'TIPO': '

### Publicaciones 

In [16]:
df_ajax.filter(pl.col("TIPO") == "Publicaciones").select(
    "ACTIVIDAD",
    "AUTORES",
    "FECHA INICIO",
    "IDENTIFICADOR",
    "SUBTIPO",
    # "CATEGORÍAS",
    "KEYWORDS",
    # "RESUMEN",
    # "FUENTE",
    # "CITACIÓN",
    # "PIDs",
    "URL",
).to_dicts()

[{'ACTIVIDAD': 'Integrating and Analysing Public Procurement Data through a Knowledge Graph: A Demonstration in a Nutshell',
  'AUTORES': 'A Soylu, O Corcho, B Elvesæter, C Badenes-Olmedo, FY Martınez,',
  'FECHA INICIO': None,
  'IDENTIFICADOR': None,
  'SUBTIPO': 'Artículo',
  'KEYWORDS': None,
  'URL': None},
 {'ACTIVIDAD': 'Repository of indexed ROs',
  'AUTORES': 'O Corcho, R Gonzalez, C Badenes, F Dong',
  'FECHA INICIO': None,
  'IDENTIFICADOR': None,
  'SUBTIPO': 'Artículo',
  'KEYWORDS': None,
  'URL': None},
 {'ACTIVIDAD': 'Repository of indexed ROs',
  'AUTORES': 'O Corcho, R Gonzalez, C Badenes, F Dong',
  'FECHA INICIO': '2015-01-01',
  'IDENTIFICADOR': None,
  'SUBTIPO': 'Artículo',
  'KEYWORDS': None,
  'URL': None},
 {'ACTIVIDAD': 'FarolApp: Live Linked Data on Light Pollution.',
  'AUTORES': 'N Mihindukulasooriya, E Gonzalez, F Serena, C Badenes, Ó Corcho',
  'FECHA INICIO': '2016-01-01',
  'IDENTIFICADOR': None,
  'SUBTIPO': 'Artículo',
  'KEYWORDS': None,
  'URL': No

### Proyectos de Investigación, Desarrollo e Innovación

In [17]:
df_ajax.filter(pl.col("TIPO") == "Proyectos I+D+I y Ayudas").select(
    "ACTIVIDAD",
    "AUTORES",
    "FECHA INICIO",
    "FECHA FIN",
    "IDENTIFICADOR",
    "SUBTIPO",
    # "CITACIÓN",
    # "PIDs",
).to_dicts()

[{'ACTIVIDAD': 'DRUGS4COVID++: Servicios de Inteligencia Artificial para la creación de un grafo de conocimientos sobre fármacos usados en el control clínico de la enfermedad, a partir de la explotación de grandes corpus de documentación científica sobre SARS-COV-2 y COVID-19',
  'AUTORES': 'Corcho García, Oscar (Investigador principal (IP)), BADENES OLMEDO, CARLOS (Miembro del equipo de investigación)',
  'FECHA INICIO': '2020-09-01',
  'FECHA FIN': '2022-10-09',
  'IDENTIFICADOR': None,
  'SUBTIPO': 'Ayudas para contratos'},
 {'ACTIVIDAD': 'Promoting Scientific Creativity by Utilising Web-based Research Objects',
  'AUTORES': 'Garijo Verdejo, Daniel (Participante), Corcho García, Oscar (Investigador principal (IP)), REDONDO GARCIA, JOSE LUIS (Participante), BADENES OLMEDO, CARLOS (Participante), IBARROLA DE ANDRÉS, ANA ANTONIA (Participante), ALCAZAR VALDARACETE, RAUL (Participante), Ramos Gargantilla, José Angel (Participante), GIRALDO PASMIN, OLGA XIMENA (Participante), PRIYATNA, F

### Trabajos en congresos

In [18]:
df_ajax.filter(
    pl.col("TIPO") == "Trabajos presentados en Congresos u Otros Eventos"
).select(
    "ACTIVIDAD",
    "AUTORES",
    "FECHA INICIO",
    # "RESUMEN",
    "IDENTIFICADOR",
    "SUBTIPO",
    # "FUENTE",
    # "CITACIÓN",
    "URL",
).to_dicts()

[{'ACTIVIDAD': 'Data analysis and mining of mobile phone dataset',
  'AUTORES': 'Carlos Badenes Olmedo, Susana Muñoz',
  'FECHA INICIO': '2013-05-01',
  'IDENTIFICADOR': None,
  'SUBTIPO': 'Comunicación',
  'URL': None},
 {'ACTIVIDAD': 'Big Data Analysis of trends in phone calls record to improve the communications',
  'AUTORES': 'Carlos Badenes Olmedo, Susana Muñoz',
  'FECHA INICIO': '2014-10-10',
  'IDENTIFICADOR': None,
  'SUBTIPO': 'Comunicación',
  'URL': None},
 {'ACTIVIDAD': 'FarolApp: Live Linked Data on Light Pollution',
  'AUTORES': 'MIHINDUKULASOORIYA, NANDANA SAMPATH, Corcho García, Oscar, González Guardia, Esteban, BADENES OLMEDO, CARLOS',
  'FECHA INICIO': '2016-10-17',
  'IDENTIFICADOR': None,
  'SUBTIPO': 'Comunicación',
  'URL': None},
 {'ACTIVIDAD': 'FarolApp: Live Linked Data on Light Pollution',
  'AUTORES': 'Nandana Mihindukulasooriya, Esteban Gonzalez, Fernando Serena, Carlos Badenes Olmedo, Oscar Corcho',
  'FECHA INICIO': '2016-10-17',
  'IDENTIFICADOR': None,


## LABORAL

### Experiencia

In [19]:
# Añadir a la sección la formación del PDI?
df_ajax.filter(pl.col("TIPO") == "Experiencia profesional").select(
    "ACTIVIDAD", "AUTORES", "FECHA INICIO", "FECHA FIN", "SUBTIPO"
).to_dicts()

[{'ACTIVIDAD': 'Software Engineer. AMPLIA SOLUCIONES, S.L.',
  'AUTORES': 'BADENES OLMEDO, CARLOS',
  'FECHA INICIO': '2006-04-01',
  'FECHA FIN': '2014-12-31',
  'SUBTIPO': 'Cargos y actividades desempeñadas anteriores a la actual'},
 {'ACTIVIDAD': 'Miembro. INTELIGENCIA ARTIFICIAL',
  'AUTORES': 'BADENES OLMEDO, CARLOS',
  'FECHA INICIO': '2016-01-01',
  'FECHA FIN': '2017-06-30',
  'SUBTIPO': 'Cargos y actividades desempeñadas anteriores a la actual'},
 {'ACTIVIDAD': 'Miembro. INTELIGENCIA ARTIFICIAL',
  'AUTORES': 'BADENES OLMEDO, CARLOS',
  'FECHA INICIO': '2018-01-01',
  'FECHA FIN': '2019-12-31',
  'SUBTIPO': 'Cargos y actividades desempeñadas anteriores a la actual'},
 {'ACTIVIDAD': 'Miembro. INTELIGENCIA ARTIFICIAL',
  'AUTORES': 'BADENES OLMEDO, CARLOS',
  'FECHA INICIO': '2020-01-01',
  'FECHA FIN': '2020-06-30',
  'SUBTIPO': 'Cargos y actividades desempeñadas anteriores a la actual'},
 {'ACTIVIDAD': 'Miembro. INTELIGENCIA ARTIFICIAL',
  'AUTORES': 'BADENES OLMEDO, CARLOS',


### Grupos, comités y foros

In [20]:
df_ajax.filter(pl.col("TIPO") == "Grupos, Redes, Sociedades, Foros y Comités").select(
    "ACTIVIDAD",
    "AUTORES",
    "FECHA INICIO",  # "CITACIÓN"
).to_dicts()

[{'ACTIVIDAD': 'Program Committees of ICTERI 2018 Main Conference, PhD Symposium, and Poster / Demo Track',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2018-01-01'},
 {'ACTIVIDAD': 'Program Committees of ICTERI 2019 Main Conference, PhD Symposium, and Poster / Demo Track',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2019-01-01'},
 {'ACTIVIDAD': 'Information Processing & Management',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2019-03-19'},
 {'ACTIVIDAD': 'SEMAPRO 2019 Technical Program Committee',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2019-06-13'},
 {'ACTIVIDAD': 'The 18th International Semantic Web Conference - Poster & Demo Track',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2019-06-27'},
 {'ACTIVIDAD': 'Masters Symposium on Advances in Data Mining, Ma

### Propiedad intelectual

In [21]:
df_ajax.filter(pl.col("TIPO") == "Propiedad Industrial e Intelectual").filter(
    pl.col("IDENTIFICADOR").is_not_null()
).select("ACTIVIDAD", "AUTORES", "IDENTIFICADOR", "URL").to_dicts()

[{'ACTIVIDAD': 'library AIssistant',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Inventores/autores/obtentores)',
  'IDENTIFICADOR': 'M-007342/2016',
  'URL': None},
 {'ACTIVIDAD': 'R4R',
  'AUTORES': 'GOMEZ PEREZ, ASUNCION DE MARIA (Autor o Coautor), BADENES OLMEDO, CARLOS (Autor o Coautor), YEDRO MARTÍNEZ, FRANCISCO LEONARDO (Autor o Coautor), Corcho García, Oscar (Autor o Coautor)',
  'IDENTIFICADOR': 'M-001472/2021',
  'URL': None}]

## DOCENCIA

### Asignaturas impartidas

In [45]:
# Pipeline optimizado de asignaturas: menos UDFs Python y más expresiones nativas de Polars
def build_subject_key(name: str, subtype: str) -> str:
    canonical = f"{name}|{subtype}"
    digest = hashlib.sha1(canonical.encode()).hexdigest()[:8]
    return f"subj_{digest}"


def normalize_subject_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name)
        .fill_null("")
        .str.strip_chars()
        .str.replace_all('"', "")
        .str.replace_all(r"\s+", " ")
        .str.to_lowercase()
        .str.to_titlecase()
    )


docencia = df_ajax.filter(pl.col("TIPO") == "Docencia Impartida").with_columns(
    [
        normalize_subject_expr("ACTIVIDAD").alias("ACTIVIDAD"),
        normalize_subject_expr("SUBTIPO").alias("SUBTIPO"),
        pl.col("FECHA INICIO")
        .str.to_date("%Y-%m-%d", strict=False)
        .alias("FECHA INICIO"),
        pl.col("FECHA FIN").str.to_date("%Y-%m-%d", strict=False).alias("FECHA FIN"),
    ]
)

max_fecha_fin = docencia.select(pl.col("FECHA FIN").max()).item()

docencia = docencia.with_columns(
    pl.when(
        (pl.col("SUBTIPO") == "Docencia Oficial Nacional")
        & pl.col("FECHA INICIO").is_not_null()
        & pl.col("FECHA FIN").is_null()
    )
    .then(pl.lit(max_fecha_fin, dtype=pl.Date))
    .otherwise(pl.col("FECHA FIN"))
    .alias("FECHA FIN")
)

asignaturas = (
    docencia.group_by(["ACTIVIDAD", "SUBTIPO"])
    .agg(
        [
            pl.col("FECHA INICIO").min().alias("FECHA INICIO"),
            pl.col("FECHA FIN").max().alias("FECHA FIN"),
        ]
    )
    .with_columns(
        pl.struct(["ACTIVIDAD", "SUBTIPO"])
        .map_elements(
            lambda x: build_subject_key(x["ACTIVIDAD"], x["SUBTIPO"]),
            return_dtype=pl.String,
        )
        .alias("_key")
    )
    .select("_key", "ACTIVIDAD", "SUBTIPO", "FECHA INICIO", "FECHA FIN")
    .sort(["ACTIVIDAD", "FECHA INICIO"])
).to_dicts()

asignaturas

[{'_key': 'subj_0a9e051b',
  'ACTIVIDAD': 'Aprendizaje Automático',
  'SUBTIPO': 'Docencia En I+D Y/O Posformación Sanitaria',
  'FECHA INICIO': datetime.date(2021, 9, 6),
  'FECHA FIN': datetime.date(2022, 1, 31)},
 {'_key': 'subj_ac1aaac9',
  'ACTIVIDAD': 'Data Science & Advanced Analytics',
  'SUBTIPO': 'Docencia Oficial Nacional',
  'FECHA INICIO': datetime.date(2019, 3, 25),
  'FECHA FIN': datetime.date(2020, 4, 24)},
 {'_key': 'subj_872089f1',
  'ACTIVIDAD': 'Descubrimiento De Conocimiento En Datos Complejos',
  'SUBTIPO': 'Docencia Oficial Nacional',
  'FECHA INICIO': datetime.date(2023, 9, 1),
  'FECHA FIN': datetime.date(2025, 8, 31)},
 {'_key': 'subj_4c7e56b0',
  'ACTIVIDAD': 'Formal Languages, Automata And Computing',
  'SUBTIPO': 'Docencia Oficial Nacional',
  'FECHA INICIO': datetime.date(2020, 10, 1),
  'FECHA FIN': datetime.date(2020, 10, 19)},
 {'_key': 'subj_24b4cb5e',
  'ACTIVIDAD': 'Fundamentos De Ingenieria De Software',
  'SUBTIPO': 'Docencia Oficial Nacional',
  '

### Dirección de Trabajos de Fin de Titulación y Tesis

In [23]:
df_ajax.filter(pl.col("TIPO") == "Dirección de Tesis, TFG, TFM, etc.").select(
    "ACTIVIDAD",
    "AUTORES",
    "FECHA INICIO",
    "SUBTIPO",
    # "CATEGORÍAS",
    # "CITACIÓN",
    # "PIDs",
).to_dicts()

[{'ACTIVIDAD': 'Explainable QA over KG',
  'AUTORES': 'Corcho García, Oscar (Director) , BADENES OLMEDO, CARLOS (Director) Autor/a: Inés Guillén, Rafael',
  'FECHA INICIO': '2021-06-01',
  'SUBTIPO': 'DEA, máster, etc.'},
 {'ACTIVIDAD': 'Named entity recognition and normalization in biomedical literature: a practical case in SARS-CoV-2 literature',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Director) , Corcho García, Oscar (Director) Autor/a: Alonso Casero, Álvaro',
  'FECHA INICIO': '2021-07-01',
  'SUBTIPO': 'DEA, máster, etc.'},
 {'ACTIVIDAD': 'Propuesta de un modelo de asociación de tópicos',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Director) , Corcho García, Oscar (Director) Autor/a: Vivanco Vásquez, Katherine',
  'FECHA INICIO': '2021-07-21',
  'SUBTIPO': 'DEA, máster, etc.'},
 {'ACTIVIDAD': 'Relation extraction between biomedical entities from scientific texts',
  'AUTORES': 'Corcho García, Oscar (Director) , BADENES OLMEDO, CARLOS (Director) Autor/a: Paneque Moreda, Christian Marlon',


In [24]:
df_ajax.filter(pl.col("TIPO") == "Proyectos de Innovacion Docente").select(
    "ACTIVIDAD", "AUTORES", "FECHA INICIO", "FECHA FIN", "URL"
).to_dicts()

[{'ACTIVIDAD': 'Técnicas innovadoras para el aprendizaje de metodologías ágiles: videojuego educativo de realidad virtual y actividad LEGO® Serious Play',
  'AUTORES': 'Mayor Márquez, Jesús, PEREZ BENEDI, JENIFER, GAYOSO CABADA, JOAQUIN, Lopez Fernandez, Daniel, BADENES OLMEDO, CARLOS, CIMMINO ARRIAGA, ANDREA JESUS',
  'FECHA INICIO': '2023-02-01',
  'FECHA FIN': '2023-11-30',
  'URL': 'https://innovacioneducativa.upm.es/proyectos-ie/informacion?anyo=2022-2023&id=756'},
 {'ACTIVIDAD': 'Desarrollo y evaluación de una herramienta para la validación automática de diagramas de modelado de software',
  'AUTORES': 'GAYOSO CABADA, JOAQUIN, Gordillo Mendez, Aldo, BADENES OLMEDO, CARLOS, Lopez Fernandez, Daniel, CIMMINO ARRIAGA, ANDREA JESUS',
  'FECHA INICIO': '2023-02-17',
  'FECHA FIN': '2023-11-30',
  'URL': None},
 {'ACTIVIDAD': 'Automatización de Evaluaciones: Desarrollo de Cuestionarios a través de Procesamiento de Lenguaje Natural y Aprendizaje Automático',
  'AUTORES': 'HERNÁNDEZ PEÑAL

### Publicaciones docentes

In [25]:
df_ajax.filter(pl.col("TIPO") == "Publicaciones Docentes").select(
    "ACTIVIDAD", "FECHA INICIO", "SUBTIPO", "URL"
).to_dicts()

[{'ACTIVIDAD': 'I Jornadas de Innovación Educativa 24 de junio de 2024: Libro de resúmenes',
  'FECHA INICIO': '2024-07-01',
  'SUBTIPO': 'Libro',
  'URL': None},
 {'ACTIVIDAD': 'Integrating Artificial Intelligence in Higher Education: Potential, Challenges and Opportunities',
  'FECHA INICIO': '2024-12-04',
  'SUBTIPO': 'Libro',
  'URL': None},
 {'ACTIVIDAD': 'Integración de la Inteligencia Artificial en la Educación Superior: Potencial, Desafíos y Oportunidades',
  'FECHA INICIO': '2024-07-04',
  'SUBTIPO': 'Otro Material Docente',
  'URL': None}]

## GESTIÓN

### Cargos de gestión

In [26]:
df_ajax.filter(pl.col("TIPO") == "Gestión").select(
    pl.col("ACTIVIDAD"),
    pl.col("FECHA INICIO"),
    pl.col("FECHA FIN"),
    pl.col("SUBTIPO"),
    # pl.col("CITACIÓN"),
).to_dicts()

[{'ACTIVIDAD': 'Hibrid techniques for Knowledge-based NLP: Knowledge graphs meet machine learning and all their friends',
  'FECHA INICIO': '2017-12-04',
  'FECHA FIN': '2017-12-06',
  'SUBTIPO': 'Organización de actividades de I+D'},
 {'ACTIVIDAD': 'Open Summer Of Code 2019',
  'FECHA INICIO': '2019-07-01',
  'FECHA FIN': '2019-07-31',
  'SUBTIPO': 'Organización de actividades de I+D'},
 {'ACTIVIDAD': 'Build a large-scale cross-lingual text search engine from scratch',
  'FECHA INICIO': '2019-11-19',
  'FECHA FIN': '2019-11-19',
  'SUBTIPO': 'Organización de actividades de I+D'},
 {'ACTIVIDAD': 'Coordinador y responsable de la asignatura Infraestructuras de Big Data',
  'FECHA INICIO': '2023-01-30',
  'FECHA FIN': '2023-07-12',
  'SUBTIPO': 'Otros Cargos Unipersonales en Universidad'}]

### Experiencia en evaluación

In [27]:
df_ajax.filter(pl.col("TIPO") == "Experiencia en Evaluación").select(
    "ACTIVIDAD", "AUTORES", "FECHA INICIO"
).to_dicts()

[{'ACTIVIDAD': 'Tribunal Defensa TFM Andrea Escolar Peña',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2022-06-13'},
 {'ACTIVIDAD': 'Tribunal Defensa TFM Gema Castillo García',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2022-06-13'},
 {'ACTIVIDAD': 'Tribunal Defensa TFM Laia Ollé Monrás',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2022-06-13'},
 {'ACTIVIDAD': 'Tribunal Defensa TFM Sandra Alonso Paz',
  'AUTORES': 'BADENES OLMEDO, CARLOS (Miembro del comite cientifico)',
  'FECHA INICIO': '2022-06-13'}]